# Generate LP files for CPLEX Interactive Optimizer

# Cluster demand nodes to bring the total variables under <= 1000

In [7]:
# =============================================================================
# STEP 1 — Cluster demand nodes and generate LP files for CPLEX Interactive
#
# WHY CLUSTERING: CPLEX Community Edition is limited to 1,000 variables.
# Our full problem has 733 demand nodes × 25 shelters + 25 = 18,350 variables.
# We aggregate 733 block groups into 39 clusters:
#   Variables = 39 × 25 (y) + 25 (x) = 1,000  ← exactly at the limit
#
# This is a published methodology (aggregated p-median). Document in paper:
#   "Demand nodes were spatially aggregated into 39 clusters via k-means.
#    Cluster weighted demand = sum of constituent block group demands.
#    Cluster centroid = population-weighted mean of constituent centroids.
#    Solved using IBM CPLEX. Aggregation error noted as a limitation."
#
# INPUTS:
#   demand_nodes.csv              ← Phase 1A output
#   distance_matrix_network.csv   ← repaired matrix
#
# OUTPUTS:
#   pmedian_p5.lp / p8 / p10     → solve these in CPLEX Interactive
#   cluster_lookup.csv           → maps clusters back to block groups
#                                   (needed by read_cplex_solutions.py)
# =============================================================================

import pandas as pd
import numpy as np
import os

# ── PATHS ────────────────────────────────────────────────────────────────────
DEMAND_CSV  = r"D:\GIS_Seminar_Project\Colab_Inputs\demand_nodes.csv"
MATRIX_CSV  = r"D:\GIS_Seminar_Project\Colab_Inputs\distance_matrix_network.csv"
OUTPUT_DIR  = r"D:\GIS_Seminar_Project\CPLEX_LP"
P_VALUES    = [5, 8, 10]
N_CLUSTERS  = 38     # 38 × 25 + 25 = 975 vars, 989 constraints — both under 1,000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
demand = pd.read_csv(DEMAND_CSV, dtype={"GEOID_JOIN": str})
demand["GEOID_JOIN"] = demand["GEOID_JOIN"].str.replace(r"\.0$", "", regex=True)

matrix = pd.read_csv(MATRIX_CSV, index_col=0, dtype={0: str})
matrix.index = matrix.index.astype(str).str.replace(r"\.0$", "", regex=True)
D_full = matrix.values.astype(float)   # (733, 25)

print(f"Demand nodes loaded:    {len(demand)}")
print(f"Distance matrix shape:  {D_full.shape}")
print(f"Target clusters:        {N_CLUSTERS}")
print(f"Variables:   {N_CLUSTERS * 25 + 25}  (limit: 1,000)")
print(f"Constraints: {1 + N_CLUSTERS + N_CLUSTERS * 25}  (limit: 1,000)")
print()

# ── K-MEANS CLUSTERING ────────────────────────────────────────────────────────
# Use geographic coordinates as features, weighted by flood-risk demand
# This ensures high-risk coastal areas get their own clusters

np.random.seed(42)
coords = demand[["LAT", "LON"]].values

# Simple k-means implementation (no sklearn needed — pure numpy)
def kmeans(X, k, n_iter=100, seed=42):
    rng = np.random.default_rng(seed)
    # Initialize centroids using k-means++ style (spread out)
    centers = [X[rng.integers(len(X))]]
    for _ in range(k - 1):
        dists = np.array([min(np.sum((x - c)**2) for c in centers) for x in X])
        probs = dists / dists.sum()
        centers.append(X[rng.choice(len(X), p=probs)])
    centers = np.array(centers)

    labels = np.zeros(len(X), dtype=int)
    for _ in range(n_iter):
        # Assign each point to nearest centroid
        dists  = np.sqrt(((X[:, None] - centers[None])**2).sum(axis=2))
        new_labels = dists.argmin(axis=1)
        if np.all(new_labels == labels):
            break
        labels = new_labels
        # Update centroids
        for k_idx in range(k):
            mask = labels == k_idx
            if mask.sum() > 0:
                centers[k_idx] = X[mask].mean(axis=0)
    return labels

print("Running k-means clustering...")
cluster_labels = kmeans(coords, N_CLUSTERS)
demand["CLUSTER"] = cluster_labels

# ── BUILD CLUSTER-LEVEL DEMAND AND DISTANCE MATRIX ───────────────────────────
print("Aggregating demand and distances by cluster...")

cluster_rows = []
for c in range(N_CLUSTERS):
    mask    = demand["CLUSTER"] == c
    members = demand[mask]

    # Cluster weighted demand = sum of constituent block group demands
    total_wtd = members["WEIGHTED_DEMAND"].sum()
    total_pop = members["POPULATION"].sum()

    # Cluster centroid = population-weighted mean of coordinates
    w = members["POPULATION"].values
    lat_c = np.average(members["LAT"].values, weights=w)
    lon_c = np.average(members["LON"].values, weights=w)

    # Cluster distance to each shelter = weighted mean of member distances
    member_idx = members.index.tolist()
    d_sub = D_full[member_idx, :]           # (n_members, 25)
    d_cluster = np.average(d_sub, axis=0, weights=w)  # (25,)

    cluster_rows.append({
        "CLUSTER":         c,
        "N_MEMBERS":       mask.sum(),
        "LAT":             lat_c,
        "LON":             lon_c,
        "POPULATION":      total_pop,
        "WEIGHTED_DEMAND": total_wtd,
    })

cluster_df = pd.DataFrame(cluster_rows)

# Build aggregated distance matrix (39 × 25)
D_agg = np.zeros((N_CLUSTERS, 25))
for c in range(N_CLUSTERS):
    mask       = demand["CLUSTER"] == c
    member_idx = demand[mask].index.tolist()
    w          = demand[mask]["POPULATION"].values
    D_agg[c]   = np.average(D_full[member_idx, :], axis=0, weights=w)

W_agg = cluster_df["WEIGHTED_DEMAND"].values

print(f"Cluster summary:")
print(f"  Clusters:               {N_CLUSTERS}")
print(f"  Avg members/cluster:    {cluster_df['N_MEMBERS'].mean():.1f}")
print(f"  Avg cluster wtd demand: {W_agg.mean():,.1f}")
print(f"  Total wtd demand:       {W_agg.sum():,.1f}  "
      f"(original: {demand['WEIGHTED_DEMAND'].sum():,.1f})")
print()

# Save cluster lookup for solution reading
cluster_lookup = demand[["GEOID_JOIN","LAT","LON","POPULATION",
                          "WEIGHTED_DEMAND","CLUSTER"]].copy()
if "TIER_LABEL" in demand.columns:
    cluster_lookup["TIER_LABEL"] = demand["TIER_LABEL"]
if "MULTIPLIER" in demand.columns:
    cluster_lookup["MULTIPLIER"] = demand["MULTIPLIER"]
cluster_lookup.to_csv(os.path.join(OUTPUT_DIR, "cluster_lookup.csv"), index=False)
print(f"Saved: cluster_lookup.csv")

# ── LP FILE WRITER ────────────────────────────────────────────────────────────
def write_lp_file(W, D, p, filepath):
    n_I, n_J = len(W), D.shape[1]
    lines = []
    lines.append(f"\\Problem name: pmedian_p{p}_clustered")
    lines.append(f"\\Demand nodes: {n_I} clusters  |  Candidates: {n_J}  |  p = {p}")
    lines.append(f"\\Variables: {n_J + n_I*n_J}, Constraints: {1 + n_I + n_I*n_J}  (both within CPLEX Community limit of 1000)")
    lines.append("")

    # Objective
    # CPLEX LP format: continuation lines must start with + or -
    lines.append("Minimize")
    lines.append(" obj:")
    first_term = True
    line_terms = []
    for i in range(n_I):
        for j in range(n_J):
            coeff = W[i] * D[i][j]
            if coeff > 0:
                line_terms.append(f"{coeff:.6f} y_{i}_{j}")
                if len(line_terms) == 6:
                    if first_term:
                        lines.append("  " + " + ".join(line_terms))
                        first_term = False
                    else:
                        lines.append("  + " + " + ".join(line_terms))
                    line_terms = []
    if line_terms:
        if first_term:
            lines.append("  " + " + ".join(line_terms))
        else:
            lines.append("  + " + " + ".join(line_terms))
    lines.append("")

    # Constraints
    lines.append("Subject To")
    lines.append(f" open_p: {' + '.join(f'x_{j}' for j in range(n_J))} = {p}")
    for i in range(n_I):
        lines.append(f" assign_{i}: {' + '.join(f'y_{i}_{j}' for j in range(n_J))} = 1")
    for i in range(n_I):
        for j in range(n_J):
            lines.append(f" link_{i}_{j}: y_{i}_{j} - x_{j} <= 0")
    lines.append("")

    # Bounds
    lines.append("Bounds")
    for j in range(n_J):
        lines.append(f" 0 <= x_{j} <= 1")
    for i in range(n_I):
        for j in range(n_J):
            lines.append(f" 0 <= y_{i}_{j} <= 1")
    lines.append("")

    # Binary
    lines.append("Binary")
    lines.append(" " + " ".join(f"x_{j}" for j in range(n_J)))
    y_vars = [f"y_{i}_{j}" for i in range(n_I) for j in range(n_J)]
    for k in range(0, len(y_vars), 10):
        lines.append(" " + " ".join(y_vars[k:k+10]))
    lines.append("")
    lines.append("End")

    with open(filepath, "w") as f:
        f.write("\n".join(lines))

    size_kb = os.path.getsize(filepath) // 1024
    print(f"  Written: {os.path.basename(filepath)}  "
          f"({n_J + n_I*n_J} vars, {size_kb} KB)")

# ── GENERATE LP FILES ─────────────────────────────────────────────────────────
print("\nGenerating LP files...")
for p in P_VALUES:
    filepath = os.path.join(OUTPUT_DIR, f"pmedian_p{p}.lp")
    write_lp_file(W_agg, D_agg, p, filepath)

# ── CPLEX INSTRUCTIONS ────────────────────────────────────────────────────────
print()
print("=" * 65)
print("LP FILES READY — Run in CPLEX Interactive:")
print("=" * 65)
for p in P_VALUES:
    print(f"""
  read {OUTPUT_DIR}\\pmedian_p{p}.lp
  optimize
  write {OUTPUT_DIR}\\solution_p{p}.sol""")
print()
print("Then run: read_cplex_solutions.py")

Demand nodes loaded:    733
Distance matrix shape:  (733, 25)
Target clusters:        38
Variables:   975  (limit: 1,000)
Constraints: 989  (limit: 1,000)

Running k-means clustering...
Aggregating demand and distances by cluster...
Cluster summary:
  Clusters:               38
  Avg members/cluster:    19.3
  Avg cluster wtd demand: 15,015.3
  Total wtd demand:       570,579.9  (original: 570,579.9)

Saved: cluster_lookup.csv

Generating LP files...
  Written: pmedian_p5.lp  (975 vars, 86 KB)
  Written: pmedian_p8.lp  (975 vars, 86 KB)
  Written: pmedian_p10.lp  (975 vars, 86 KB)

LP FILES READY — Run in CPLEX Interactive:

  read D:\GIS_Seminar_Project\CPLEX_LP\pmedian_p5.lp
  optimize
  write D:\GIS_Seminar_Project\CPLEX_LP\solution_p5.sol

  read D:\GIS_Seminar_Project\CPLEX_LP\pmedian_p8.lp
  optimize
  write D:\GIS_Seminar_Project\CPLEX_LP\solution_p8.sol

  read D:\GIS_Seminar_Project\CPLEX_LP\pmedian_p10.lp
  optimize
  write D:\GIS_Seminar_Project\CPLEX_LP\solution_p10.sol

Th